In [5]:
import os
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, ArrayType
)
from pyspark.sql.functions import to_timestamp, col, explode

In [2]:
load_dotenv()

True

In [ ]:
url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&topics=manufacturing,technology&apikey={os.getenv('ALPHA_VANTAGE_API_KEY')}"
r = requests.get(url)
data = r.json()

{'items': '50', 'sentiment_score_definition': 'x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish', 'relevance_score_definition': '0 < x <= 1, with a higher score indicating higher relevance.', 'feed': [{'title': 'Covestor Ltd Lowers Stock Holdings in Flex Ltd. $FLEX', 'url': 'https://www.marketbeat.com/instant-alerts/filing-covestor-ltd-lowers-stock-holdings-in-flex-ltd-flex-2026-05-24/', 'time_published': '20260524T073650', 'authors': ['MarketBeat'], 'summary': 'Covestor Ltd significantly reduced its stake in Flex Ltd. by 89.6% in the fourth quarter, ending up with 1,550 shares valued at $94,000. Despite this, institutional investors still hold 94.30% of the company\'s shares. Flex maintains a "Moderate Buy" consensus rating from analysts with an average price target of $112.80, and some analysts have recently raised their price targets significantly.', 'banner_image': 'https://www.marketbeat.co

In [6]:
# Inicia a SparkSession (local)
spark = SparkSession.builder \
    .appName("AlphaVantageNewsSentiment") \
    .master("local[*]") \
    .getOrCreate()

# --- Sub-schemas ---

topic_schema = StructType([
    StructField("topic",           StringType(), True),
    StructField("relevance_score", FloatType(),  True),
])

ticker_schema = StructType([
    StructField("ticker",                 StringType(), True),
    StructField("relevance_score",        FloatType(),  True),
    StructField("ticker_sentiment_score", FloatType(),  True),
    StructField("ticker_sentiment_label", StringType(), True),
])

# --- Schema principal ---

schema = StructType([
    StructField("title",                   StringType(),              True),
    StructField("url",                     StringType(),              True),
    StructField("time_published",          StringType(),              True),
    StructField("authors",                 StringType(),              True),
    StructField("source",                  StringType(),              True),
    StructField("source_domain",           StringType(),              True),
    StructField("summary",                 StringType(),              True),
    StructField("overall_sentiment_score", FloatType(),               True),
    StructField("overall_sentiment_label", StringType(),              True),
    StructField("topics",                  ArrayType(topic_schema),   True),
    StructField("ticker_sentiment",        ArrayType(ticker_schema),  True),
])

# --- Monta as linhas ---

feed = data.get('feed', [])

rows = []
for article in feed:
    topics = [
        Row(
            topic=t.get('topic'),
            relevance_score=float(t.get('relevance_score', 0.0)),
        )
        for t in article.get('topics', [])
    ]

    tickers = [
        Row(
            ticker=t.get('ticker'),
            relevance_score=float(t.get('relevance_score', 0.0)),
            ticker_sentiment_score=float(t.get('ticker_sentiment_score', 0.0)),
            ticker_sentiment_label=t.get('ticker_sentiment_label'),
        )
        for t in article.get('ticker_sentiment', [])
    ]

    row = (
        article.get('title'),
        article.get('url'),
        article.get('time_published'),
        ', '.join(article.get('authors', [])),
        article.get('source'),
        article.get('source_domain'),
        article.get('summary'),
        article.get('overall_sentiment_score'),
        article.get('overall_sentiment_label'),
        topics,
        tickers,
    )
    rows.append(row)

# --- Cria o DataFrame PySpark ---

df = spark.createDataFrame(rows, schema=schema)

df = df.withColumn(
    "time_published",
    to_timestamp(col("time_published"), "yyyyMMdd'T'HHmmss")
)

print(f"Total de artigos: {df.count()}")
df.printSchema()

Total de artigos: 50

Colunas: ['title', 'url', 'time_published', 'authors', 'source', 'source_domain', 'summary', 'overall_sentiment_score', 'overall_sentiment_label', 'topics', 'tickers']


In [7]:
df.show(5, truncate=80)

,title,url,time_published,authors,source,source_domain,summary,overall_sentiment_score,overall_sentiment_label,topics,tickers
0,Covestor Ltd Lowers Stock Holdings in Flex Ltd...,https://www.marketbeat.com/instant-alerts/fili...,2026-05-24 07:36:50,MarketBeat,MarketBeat,MarketBeat,Covestor Ltd significantly reduced its stake i...,0.199490,Somewhat-Bullish,"earnings, financial_markets, finance, technolo...",FLEX
1,Honeywell Grant And Army Deal Highlight Quantu...,https://simplywall.st/stocks/us/capital-goods/...,2026-05-24 06:43:02,"Simply Wall St, Bailey Pemberton",Simply Wall Street,Simply Wall Street,"Honeywell's quantum computing subsidiary, Quan...",0.307188,Somewhat-Bullish,"manufacturing, technology, financial_markets, ...",HON
2,Horizon Aircraft Advances Dual-Use Certificati...,https://www.aero-news.net/index.cfm?do=main.te...,2026-05-24 04:09:14,,Aero-News Network,Aero-News Network,Horizon Aircraft is making progress on the dua...,0.445192,Bullish,"energy_transportation, manufacturing, technology",HOVR
3,"Viasat, Intelsat Win $438M Air Force Satellite...",https://news.clearancejobs.com/2026/05/23/vias...,2026-05-24 01:39:55,Jillian Hamilton,Clearance Jobs,Clearance Jobs,Viasat Inc. and Intelsat General Communication...,0.311115,Somewhat-Bullish,"technology, economy_fiscal, manufacturing","VSAT, LHX, HII, LMT"
4,Elbit signs deal with German submarine manufac...,https://www.jpost.com/israel-news/defense-news...,2026-05-23 23:39:53,UDI ETZION,The Jerusalem Post,The Jerusalem Post,Elbit Systems and German submarine manufacture...,0.369216,Bullish,"manufacturing, technology, finance","ESLT, GD"


In [8]:
# Equivalente ao .unique() do pandas — usa .distinct() no PySpark
df.select("overall_sentiment_label").distinct().show()

array(['Somewhat-Bullish', 'Bullish', 'Neutral', 'Somewhat-Bearish',
       'Bearish'], dtype=object)

In [ ]:
# Explode tickers: uma linha por ticker por artigo
df_tickers = df.select(
    "title",
    "time_published",
    "overall_sentiment_label",
    explode("ticker_sentiment").alias("t")
).select(
    "title",
    "time_published",
    "overall_sentiment_label",
    col("t.ticker").alias("ticker"),
    col("t.ticker_sentiment_score").alias("sentiment_score"),
    col("t.ticker_sentiment_label").alias("sentiment_label"),
)

df_tickers.show(10, truncate=60)

In [ ]:
# Explode topics: uma linha por tópico por artigo
df_topics = df.select(
    "title",
    "overall_sentiment_score",
    explode("topics").alias("tp")
).select(
    "title",
    "overall_sentiment_score",
    col("tp.topic").alias("topic"),
    col("tp.relevance_score").alias("topic_relevance"),
)

df_topics.show(10, truncate=60)